[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/shripada/ame5003-nlp/blob/main/labs/lab-05-ngram-language-models.ipynb)

**Click the badge above to open this lab in Google Colab.** Then choose *File → Save a copy in Drive* so your work is saved.

# Lab 5 — N-grams and n-gram probabilities

**MSIS · AME 5053 · Week 5 · 3 hours**

Unit I was about finding text. This lab is the first one where you build something
that **predicts** text.

You will build an n-gram language model twice. First by hand, on the four-sentence
corpus from sessions 11–13, small enough that you can check every number against the
lesson. Then on a real book, where you will watch the same code produce English.

**By the end of this lab you will be able to:**

1. Count unigrams and bigrams from a corpus
2. Turn those counts into probabilities with the MLE formula
3. Score a whole sentence, and see an unseen pair drive it to zero
4. Apply add-one smoothing, and measure what it costs
5. Generate text by sampling from the model
6. Check your work against NLTK's own language model

Run every cell, in order.

---
## Part 0 — Setup

NLTK is already installed on Colab, but its **corpora are not** — those download
separately, and they vanish when Colab recycles the machine. So this cell runs every
session.

In [ ]:
!pip install -q nltk regex

import nltk

# No quiet=True: if a download fails we want to see it, not carry on with no data.
ok = nltk.download("gutenberg")
print("gutenberg downloaded:", ok)

from nltk.corpus import gutenberg
print("books available:", len(gutenberg.fileids()))
print("Done.")

> **Save your own copy now:** File → Save a copy in Drive.

If `gutenberg downloaded: False` above, re-run the cell — it is almost always a network
hiccup. Nothing after Part 3 will work without it.

---
## Part 1 — Build the model on a corpus you can check by hand

These are the four sentences from sessions 11–13. Four sentences is uselessly small for a
real model and perfect for checking that your code is right.

`<s>` and `</s>` mark the start and end of each sentence. They are counted as tokens:
`<s>` gives the first word something to condition on, and `</s>` is how the model knows
when to stop.

In [ ]:
corpus = [
    "I want to eat Indian food",
    "I want to eat Chinese food",
    "I want Indian food",
    "she wants to eat Indian food",
]

# From here on the markers are ordinary tokens: they are counted like words,
# and they form bigrams like ("<s>", "I") and ("food", "</s>").
sents = [["<s>"] + s.split() + ["</s>"] for s in corpus]
for s in sents:
    print(" ".join(s))

Now count. A `Counter` over the tokens gives unigram counts; a `Counter` over
`zip(s, s[1:])` gives bigram counts — that `zip` **is** the sliding window from session
11, in one line.

In [ ]:
from collections import Counter

unigrams = Counter()
bigrams = Counter()

# Counter.update ADDS to the running totals, so this accumulates across all
# four sentences rather than starting again on each one.
for s in sents:
    unigrams.update(s)              # every token, markers included
    # zip(s, s[1:]) pairs each token with the one after it. For
    # ["<s>", "I", "want"] that is ("<s>", "I") and ("I", "want") — the
    # sliding window of session 11, and it stops on its own at the last token.
    bigrams.update(zip(s, s[1:]))

print("unigram counts")
for w, c in sorted(unigrams.items()):
    print(f"  {w:8s} {c}")

print("\nbigram counts")
for (a, b), c in sorted(bigrams.items()):
    print(f"  {a:8s} {b:8s} {c}")

# Verified output:
#   unigram counts
#     </s>     4
#     <s>      4
#     Chinese  1
#     I        3
#     Indian   3
#     eat      3
#     food     4
#     she      1
#     to       3
#     want     3
#     wants    1
#   bigram counts (13 distinct pairs)
#     <s> I 3 · <s> she 1 · Chinese food 1 · I want 3 · Indian food 3
#     eat Chinese 1 · eat Indian 2 · food </s> 4 · she wants 1 · to eat 3
#     want Indian 1 · want to 2 · wants to 1

### From counts to probabilities

The maximum likelihood estimate, straight from session 12:

```
P(w2 | w1) = count(w1 w2) / count(w1)
```

We use `Fraction` rather than `float` so the output is `2/3`, not `0.6666666666666666`.
Easier to check against the lesson.

In [ ]:
from fractions import Fraction

def P(w2, w1):
    """MLE estimate of P(w2 | w1)."""
    # A context we never saw has no distribution at all, and dividing by its
    # count of 0 would raise. Return 0 and let the caller notice.
    if unigrams[w1] == 0:
        return Fraction(0)
    # A Counter returns 0 for a key it does not hold, so an unseen pair needs
    # no special case here — it simply gives a numerator of 0. That is Part 2.
    return Fraction(bigrams[(w1, w2)], unigrams[w1])

print("P(Indian  | eat)  =", P("Indian", "eat"))
print("P(Chinese | eat)  =", P("Chinese", "eat"))
print("P(to      | want) =", P("to", "want"))

# Verified output:
#   P(Indian  | eat)  = 2/3
#   P(Chinese | eat)  = 1/3
#   P(to      | want) = 2/3

`2/3` and `1/3` — the numbers from the board. Note they add to 1: those are the only two
words that ever followed `eat`, so the model's distribution over next words is complete.

### Scoring a whole sentence

Multiply the bigram probabilities along the sentence, including the two markers.

In [ ]:
def score(sentence):
    """Bigram probability of a whole sentence."""
    # Pad exactly as the training sentences were padded, or the first and last
    # words would be scored against contexts the model was never given.
    toks = ["<s>"] + sentence.split() + ["</s>"]
    p = Fraction(1)                  # 1 is the identity for the product below
    # Same sliding window as the counting cell, so the chain includes
    # P(I | <s>) at the start and P(</s> | food) at the end.
    for a, b in zip(toks, toks[1:]):
        p *= P(b, a)
    return p

for s in corpus:
    print(f"{s:32s} {str(score(s)):5s} = {float(score(s)):.4f}")

# These are the exact values from session 12. If any assert fails,
# the bug is in your code, not in the lesson.
assert score("I want to eat Indian food") == Fraction(1, 3)
assert score("I want to eat Chinese food") == Fraction(1, 6)
assert score("she wants to eat Indian food") == Fraction(1, 6)
print("\nAll assertions passed — your model matches the lesson.")

# Verified output:
#   I want to eat Indian food        1/3   = 0.3333
#   I want to eat Chinese food       1/6   = 0.1667
#   I want Indian food               1/4   = 0.2500
#   she wants to eat Indian food     1/6   = 0.1667
#   All assertions passed — your model matches the lesson.

The model ranks `Indian food` above `Chinese food`, and it has no idea what either phrase
means. It only counted.

### Your turn

Score the sentence `I want Indian food` and confirm it comes to **1/4**. Then say in one
line why it scores *lower* than the longer `I want to eat Indian food`.

In [ ]:
# YOUR CODE HERE
# Call score() on the sentence and compare with Fraction(1, 4).

---
## Part 2 — Then it breaks

Score an ordinary English sentence that the corpus happens not to contain.

In [ ]:
# Every word here is in the vocabulary. It is the *pair* that is new.
bad = "I want to eat Indian pizza"
print(bad, "->", score(bad))
print("count('Indian pizza') =", bigrams[("Indian", "pizza")])
print("P(pizza | Indian)     =", P("pizza", "Indian"))

assert score(bad) == 0

# Verified output:
#   I want to eat Indian pizza -> 0
#   count('Indian pizza') = 0
#   P(pizza | Indian)     = 0

**Zero.** Not unlikely — impossible. Five of the six words were fine; one unseen pair
multiplied the whole product to nothing.

This is not a small-corpus problem that more data fixes. You will measure exactly how bad
it gets on a real book in Part 4.

---
## Part 3 — Add-one smoothing, and what it costs

Session 13's first fix: pretend every possible pair was seen one extra time.

```
P_add1(w2 | w1) = (count(w1 w2) + 1) / (count(w1) + V)
```

`V` is the number of tokens the model can **predict**. That includes `</s>`, which the
model does predict, and excludes `<s>`, which only ever appears as context.

In [ ]:
# One flat set of every token the model can emit. `<s>` is filtered out
# because it only ever appears to the right of the bar in P(w2 | w1) —
# nothing is ever predicted to be a sentence start.
V_tokens = sorted(set(w for s in sents for w in s if w != "<s>"))
V = len(V_tokens)

print("predictable vocabulary:", V_tokens)
print("V =", V)

# Verified output:
#   predictable vocabulary: ['</s>', 'Chinese', 'I', 'Indian', 'eat', 'food',
#                            'she', 'to', 'want', 'wants']
#   V = 10

In [ ]:
def P_add1(w2, w1, V=V):
    """Add-one (Laplace) smoothed estimate of P(w2 | w1)."""
    # +1 on top gives every pair a count it may not have earned, so no
    # numerator is ever 0. The +V below is that same 1 added once for each of
    # the V words it was given to, which is what keeps the row summing to 1.
    return Fraction(bigrams[(w1, w2)] + 1, unigrams[w1] + V)

print(f"{'after eat':10s} {'count':>5s} {'MLE':>8s} {'add-one':>10s}")
for w in ["Indian", "Chinese", "food", "to"]:
    print(f"{w:10s} {bigrams[('eat', w)]:5d} {str(P(w, 'eat')):>8s} {str(P_add1(w, 'eat')):>10s}")

assert P_add1("Indian", "eat") == Fraction(3, 13)
assert P_add1("food", "Indian") == Fraction(4, 13)
print("\nMatches session 13.")

# Verified output:
#   after eat  count      MLE    add-one
#   Indian         2      2/3       3/13
#   Chinese        1      1/3       2/13
#   food           0        0       1/13
#   to             0        0       1/13
#   Matches session 13.

Nothing is zero any more, and the ordering survives. But look at the price:
`P(Indian | eat)` fell from **2/3 = 0.667** to **3/13 = 0.231**. Two-thirds of the
evidence you actually collected was handed to words that never once followed `eat`.

Now watch that get worse as the vocabulary grows to a realistic size. `P(food | Indian)`
is a pair seen in **3 out of 3** opportunities:

In [ ]:
print("MLE P(food | Indian) =", P("food", "Indian"), "\n")
# count("Indian food") is 3 and count("Indian") is 3, so the smoothed value is
# (3 + 1) / (3 + V). Only V changes across the three lines below.
for Vr in (10, 1000, 50000):
    print(f"  V = {Vr:>6}   add-one = 4/{3+Vr:<6} = {4/(3+Vr):.6f}")

# Verified output:
#   MLE P(food | Indian) = 1
#     V =     10   add-one = 4/13     = 0.307692
#     V =   1000   add-one = 4/1003   = 0.003988
#     V =  50000   add-one = 4/50003  = 0.000080

At a realistic vocabulary, add-one takes a pair you observed **every single time** and
rates it at eight thousandths of one percent. The 50,000 words that never followed
`Indian` soak up essentially all the probability, because add-one gives every one of them
the same share.

That flat share is the flaw, and it is why backoff and interpolation exist — they fall
back on a shorter context, which still carries evidence about which words are actually
common.

### Your turn

Compute add-one probabilities for the context `want` (which occurs 3 times, followed by
`to` twice and `Indian` once). Confirm that the smoothed values over **all** V possible
next words sum to exactly 1 — that is the constraint the `+ V` exists to satisfy.

In [ ]:
# YOUR CODE HERE
# Loop over V_tokens, print P_add1(w, "want") for each, and sum them.

---
## Part 4 — A real corpus

The code you wrote in Part 1 is finished. It does not need to change to handle 27,000
words instead of 26 — only the data changes.

We use *Alice's Adventures in Wonderland* from the Gutenberg corpus. `gutenberg.words()`
is **already tokenized**, so there is no tokenizer to configure. We lowercase, and keep
alphabetic tokens only, to drop punctuation.

In [ ]:
# .lower() so that "The" and "the" are one word rather than two, and
# .isalpha() to drop punctuation, which gutenberg.words() returns as separate
# tokens. It drops digits too — a rough filter, but Alice has almost none.
words = [w.lower() for w in gutenberg.words("carroll-alice.txt") if w.isalpha()]

print("tokens         :", len(words))
print("vocabulary size:", len(set(words)))
print("first 12 tokens:", words[:12])

# Verified output:
#   tokens         : 27333
#   vocabulary size: 2569
#   first 12 tokens: ['alice', 's', 'adventures', 'in', 'wonderland', 'by',
#                     'lewis', 'carroll', 'chapter', 'i', 'down', 'the']

In [ ]:
# The same two lines as Part 1, with no loop because the book is one long
# stream of words. There are no sentence markers, so a bigram here can straddle
# a full stop — acceptable for counting, and it is why Part 5 never stops.
g_uni = Counter(words)
g_bi = Counter(zip(words, words[1:]))

print("most common words  :", g_uni.most_common(5))
print("most common bigrams:", g_bi.most_common(5))

# Verified output:
#   most common words  : [('the', 1642), ('and', 872), ('to', 729), ('a', 632), ('it', 595)]
#   most common bigrams: [(('said', 'the'), 210), (('of', 'the'), 133),
#                         (('said', 'alice'), 116), (('in', 'a'), 97), (('and', 'the'), 82)]

`said the` and `said alice` at the top is the book's own voice showing up in the counts —
*Alice* is mostly dialogue. A model trained here will sound like it.

Now real conditional probabilities. Notice how much smaller they are than the toy
corpus's `2/3`: with 2,569 words available, no single continuation dominates.

In [ ]:
def gP(w2, w1):
    """Same MLE formula as Part 1, different counts."""
    # float rather than Fraction: 20/398 tells us nothing as a fraction, and
    # exact arithmetic over 27,000 tokens is slow for no gain.
    return g_bi[(w1, w2)] / g_uni[w1] if g_uni[w1] else 0.0

print("count('alice') =", g_uni["alice"], "\n")
for w in ["and", "was", "thought", "said"]:
    print(f"  P({w:8s} | alice) = {g_bi[('alice', w)]:3d} / {g_uni['alice']} = {gP(w, 'alice'):.4f}")

# Keep only the bigrams whose FIRST word is "alice", then index them by the
# second word — the whole distribution over what follows, ranked.
nxt = Counter({b: c for (a, b), c in g_bi.items() if a == "alice"})
print("\ntop continuations of 'alice':", nxt.most_common(6))

# Verified output:
#   count('alice') = 398
#     P(and      | alice) =  20 / 398 = 0.0503
#     P(was      | alice) =  17 / 398 = 0.0427
#     P(thought  | alice) =  12 / 398 = 0.0302
#     P(said     | alice) =  11 / 398 = 0.0276
#   top continuations of 'alice': [('and', 20), ('was', 17), ('i', 16),
#                                  ('s', 12), ('thought', 12), ('had', 11)]

**This is what a real distribution looks like.** The best guess after `alice` gets 5% —
not 100%, as the toy corpus suggested. If you took `P(food | Indian) = 1.0` in Part 1 as
the model being confident, this is the correction.

Now measure the zero problem properly. How many of the possible bigrams does a whole book
actually contain?

In [ ]:
seen = len(g_bi)                      # distinct pairs with a count of 1 or more
possible = len(set(words)) ** 2       # any word may follow any word: V x V

print(f"distinct bigrams seen: {seen:>12,}")
print(f"possible bigrams     : {possible:>12,}")
print(f"fraction ever seen   : {seen / possible:.4%}")

# Verified output:
#   distinct bigrams seen:       14,514
#   possible bigrams     :    6,599,761
#   fraction ever seen   : 0.2199%

**Two tenths of one percent.** A whole book, and 99.78% of the bigrams it could contain
have a count of zero — each one a sentence this model would call impossible.

And this is the *easy* case. A larger corpus has a larger vocabulary, so `possible` grows
faster than `seen` does. More data does not fix this; it is why smoothing is not optional.

---
## Part 5 — Generating text

A language model predicts the next word. Do that repeatedly and it writes.

We **sample** from the distribution rather than always taking the most likely word.
Always taking the top choice gets stuck in a loop almost immediately — `the` → `the` →
`the`. Sampling with `random.choices`, weighted by the counts, means a word twice as
common is picked twice as often.

Every cell here is **seeded**, so your output matches the comments. Change the seed and
you get different text — that is the model working, not a bug.

In [ ]:
import random
from collections import defaultdict

# g_bi is keyed by pairs, which is the wrong shape for generation: we have a
# word and need its continuations. Reindex once, into word -> Counter of the
# words that followed it, so each step of the loop below is a dict lookup.
follow = defaultdict(Counter)
for (a, b), c in g_bi.items():
    follow[a][b] = c

def generate(start, n=12, seed=0):
    random.seed(seed)                # fixed seed, so this cell is reproducible
    out = [start]
    for _ in range(n - 1):           # the start word is already one of the n
        opts = follow[out[-1]]       # condition on the last word, and only that
        if not opts:
            break                    # nothing ever followed it: stop early
        # weights are the raw counts, so a continuation seen twice as often is
        # chosen twice as often. random.choices returns a one-item list.
        out.append(random.choices(list(opts), weights=list(opts.values()))[0])
    return " ".join(out)

for sd in (0, 1, 2):
    print(f"seed {sd}: {generate('alice', 12, sd)}")

# Verified output:
#   seed 0: alice he taught us all i ll go alice very unpleasant state
#   seed 1: alice soon had the duchess chop off with his head would all
#   seed 2: alice feeling at the air and shouted in the cheshire cat or

Read those carefully. `the duchess chop off with his head` is not a sentence, but every
*pair* in it is real English from the book. That is exactly what a bigram model knows:
each neighbouring pair, and nothing about the sentence as a whole.

For contrast, here is a **unigram** model — no context at all, just words drawn by
frequency:

In [ ]:
random.seed(0)
# k=12 words drawn independently by frequency. No previous word is consulted,
# which is the entire difference between this line and generate().
print("unigram:", " ".join(random.choices(list(g_uni), weights=list(g_uni.values()), k=12)))
print("bigram :", generate("alice", 12, 0))

# Verified output:
#   unigram: old fond did she not there fan it then end follow like
#   bigram : alice he taught us all i ll go alice very unpleasant state

The unigram output is word salad — real words, no structure, because it has no context.
One word of context already buys grammatical-sounding fragments.

So add another. A **trigram** model conditions on the previous *two* words:

In [ ]:
# A third offset copy of the list widens the window from pairs to triples.
g_tri = Counter(zip(words, words[1:], words[2:]))

# Same reindex as before, except the key is now a PAIR of words. That pair is
# the context, and there are far more distinct pairs than distinct words —
# which is the sparsity Part 4 measured, arriving here as a cost.
tri_follow = defaultdict(Counter)
for (a, b, c), n in g_tri.items():
    tri_follow[(a, b)][c] = n

def generate3(w1, w2, n=12, seed=0):
    random.seed(seed)
    out = [w1, w2]                   # a trigram model needs two words to start
    for _ in range(n - 2):
        opts = tri_follow[(out[-2], out[-1])]   # the last two words, together
        if not opts:
            break
        out.append(random.choices(list(opts), weights=list(opts.values()))[0])
    return " ".join(out)

for pair in [("alice", "was"), ("the", "queen"), ("said", "the")]:
    print(f"{str(pair):18s} {generate3(pair[0], pair[1], 12, 0)}")

# Verified output:
#   ('alice', 'was')   alice was too late to wish that she added in a very
#   ('the', 'queen')   the queen till she was now about two feet high i wish
#   ('said', 'the')    said the gryphon do you know he was going to dive in

**`alice was too late to wish that she added in a very`** — that is close to real
English, and the model is still only counting.

This is the trade-off from session 11, now visible: unigram is nonsense, bigram is
fragments, trigram is nearly fluent. More context, better output. And the reason we
cannot simply keep going is the sparsity you measured in Part 4 — with only 27,000 words,
most trigram contexts occur once, so the model has one choice and starts reciting the
book verbatim rather than composing.

### Your turn

Count how many trigram contexts in this corpus have exactly **one** possible continuation.
Those are the contexts where the model has no choice at all and simply copies the book.

In [ ]:
# YOUR CODE HERE
# tri_follow maps a (w1, w2) context to a Counter of continuations.
# How many of those Counters have length 1?

---
## Part 6 — Check your work against NLTK

You built all of this by hand, which is the point. But NLTK ships language models in
`nltk.lm`, and a good habit is to check hand-written code against a library.

We fit NLTK's `MLE` model to the same four sentences and compare.

In [ ]:
from nltk.lm import MLE
from nltk.lm.preprocessing import padded_everygram_pipeline

# padded_everygram_pipeline does Part 1's preparation in one call: it adds the
# <s> and </s> markers and yields every n-gram up to order 2 — unigrams and
# bigrams both, since a bigram model needs the unigram counts as denominators.
train, vocab = padded_everygram_pipeline(2, [s.split() for s in corpus])
model = MLE(2)                       # 2 = bigram

# fit() is the counting step, and nothing more: it walks the n-grams in `train`
# and tallies them, exactly as Part 1's `for s in sents:` loop did. No
# probability is worked out here. An MLE model divides one count by another
# on demand, when .score() is called, which is why fitting is instant.
#
# `vocab` is a separate argument because the model has to know the word list
# in its own right, not just infer it from what it counted. It uses that list
# to decide whether a word is known: anything outside it becomes <UNK> at
# scoring time. That is the extra slot Part 6's disagreement turns on.
#
# `train` is a generator, and fit() has now consumed it. Fitting a second model
# needs a fresh one, which is why the next cell calls the pipeline again.
model.fit(train, vocab)

# NLTK takes the context as a LIST, because the same call has to work for a
# trigram model, where the context is two words.
print("NLTK  P(Indian | eat) =", model.score("Indian", ["eat"]))
print("ours  P(Indian | eat) =", float(P("Indian", "eat")))
print()
print("NLTK  P(pizza  | Indian) =", model.score("pizza", ["Indian"]))

assert abs(model.score("Indian", ["eat"]) - 2 / 3) < 1e-12
print("\nOur MLE agrees with NLTK exactly.")

# Verified output:
#   NLTK  P(Indian | eat) = 0.6666666666666666
#   ours  P(Indian | eat) = 0.6666666666666666
#   NLTK  P(pizza  | Indian) = 0.0
#   Our MLE agrees with NLTK exactly.

Same numbers, including the zero. Now try the same comparison for **smoothing**, and
something odd happens.

In [ ]:
from nltk.lm import Laplace

# Rebuilt, not reused: the generator from the previous cell is exhausted, and
# fitting on it would silently produce a model trained on nothing.
train2, vocab2 = padded_everygram_pipeline(2, [s.split() for s in corpus])
lap = Laplace(2)                     # NLTK's name for add-one smoothing
lap.fit(train2, vocab2)

print("NLTK Laplace P(Indian | eat) =", lap.score("Indian", ["eat"]))
print("our add-one  P(Indian | eat) =", float(P_add1("Indian", "eat")), f"({P_add1('Indian', 'eat')})")
print()
print("NLTK vocabulary size:", len(lap.vocab))
print("NLTK vocabulary     :", sorted(lap.vocab))
print("our V               :", V)

# Verified output:
#   NLTK Laplace P(Indian | eat) = 0.2
#   our add-one  P(Indian | eat) = 0.23076923076923078 (3/13)
#   NLTK vocabulary size: 12
#   NLTK vocabulary     : ['</s>', '<UNK>', '<s>', 'Chinese', 'I', 'Indian',
#                          'eat', 'food', 'she', 'to', 'want', 'wants']
#   our V               : 10

**The two disagree — and neither is wrong.**

NLTK's vocabulary has **12** entries where ours has 10. It includes two extra:

- `<s>` — we excluded it because the model never predicts it, only conditions on it.
- `<UNK>` — the unknown-word token from session 13. NLTK always reserves a slot so it can
  handle a word it has never seen.

So NLTK computes `3 / (3 + 12) = 3/15 = 0.2`, and we compute `3 / (3 + 10) = 3/13 ≈
0.231`. Same formula, different `V`.

**The lesson is that `V` is a modelling choice, not a fact about the corpus.** Reserving
an `<UNK>` slot is defensible and so is leaving it out; counting `<s>` is harder to
defend, but it costs almost nothing on a real vocabulary of 50,000. What matters is that
you know which choice your code made — because if you compare two models' numbers without
checking, you will be comparing nothing at all.

This is worth remembering beyond this lab. Most "the library disagrees with my code"
moments in ML are a definition mismatch like this one, not an arithmetic error.

---
## What you built

1. **Counting** — `Counter` over tokens and over `zip(s, s[1:])`. That zip is the sliding
   window, and it is the whole of n-gram extraction.
2. **MLE probabilities** — `count(w1 w2) / count(w1)`, verified against the lesson's
   exact fractions.
3. **The zero problem** — measured, not asserted: 99.78% of possible bigrams never occur
   in a whole book.
4. **Add-one smoothing** — and the proof that it is too blunt to use on a real
   vocabulary.
5. **Generation** — unigram word salad, bigram fragments, trigram near-fluency. The
   context trade-off, visible.
6. **A disagreement with NLTK** — resolved by understanding that `V` is a choice.

The model you built here is the same *kind* of object as the one behind autocomplete and
ChatGPT: give it words, get a probability for the next one. What changes from here is how
the probabilities are worked out — counting gives way to neural networks in Unit III.

**Next lecture (session 14):** how do you tell whether a language model is any good?
Bigram or trigram, add-one or interpolation — you cannot choose without a way to measure.
That measure is perplexity.